In [ ]:
import os
import sys
# add path to custom functions
module_path = os.path.abspath(os.path.join('.')) #../..
if module_path not in sys.path:
    sys.path.append(module_path+"/scripts/py_functions")
# import custom functions
from map_plot_tools import *
from line_plot_tools import *
from colorbar_funcs import *
from data_funcs import *

import xarray as xr
xr.set_options(keep_attrs=True)
import numpy as np
#np.set_printoptions(threshold=np.inf) # disable truncation
import metpy.calc as mp
import pandas as pd 
from scipy.stats import pearsonr

import cartopy
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
from shapely.geometry.polygon import LinearRing

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.offsetbox import AnchoredText
import matplotlib.gridspec as gridspec
import matplotlib.ticker as mticker
import matplotlib.colors as mcolors
from matplotlib.colors import TwoSlopeNorm
from matplotlib import cm
from matplotlib.colors import ListedColormap,LinearSegmentedColormap
import cmocean.cm as cmo
import seaborn as sns
# settings
%config InlineBackend.figure_format = 'retina'

# top level data directory
dpath0='/glade/work/dervlamk'
# save figs here
opath='/glade/work/dervlamk'

In [ ]:
# File Paths

files = {}

for sim in ['pi']:
    files[sim] = {}
    for varn in ['PRECRC_H2Or', 'PRECRL_H2OR', 'PRECSC_H2Os', 'PRECSL_H2OS', 'PRECRC_HDOr', 'PRECRL_HDOR', 'PRECSC_HDOs', 'PRECSL_HDOS']:
        files[sim][varn] = f'{dpath0}/PI/dh.precIsotopes.atm.iPI.nc'
    for varn in ['PRECRC_H216Or', 'PRECRL_H216OR', 'PRECSC_H216Os', 'PRECSL_H216OS', 'PRECRC_H218Or', 'PRECRL_H218OR', 'PRECSC_H218Os', 'PRECSL_H218OS']:
        files[sim][varn] = f'{dpath0}/PI/o.precIsotopes.atm.iPI.nc'
    for varn in ['PRECC', 'PRECL']:
        files[sim][varn] = f'{dpath0}/PI/atm.2d.vars.PI.climo.nc'

for sim in ['lig']:
    files[sim] = {}
    for varn in ['PRECRC_H2Or', 'PRECRL_H2OR', 'PRECSC_H2Os', 'PRECSL_H2OS', 'PRECRC_HDOr', 'PRECRL_HDOR', 'PRECSC_HDOs', 'PRECSL_HDOS']:
        files[sim][varn] = f'{dpath0}/LIG/climo/dh.precIsotopes.atm.iLIG127K.climo.nc'
    for varn in ['PRECRC_H216Or', 'PRECRL_H216OR', 'PRECSC_H216Os', 'PRECSL_H216OS', 'PRECRC_H218Or', 'PRECRL_H218OR', 'PRECSC_H218Os', 'PRECSL_H218OS']:
        files[sim][varn] = f'{dpath0}/LIG/climo/o.precIsotopes.atm.iLIG127K.climo.nc'
    for varn in ['PRECC', 'PRECL']:
        files[sim][varn] = f'{dpath0}/LIG/climo/atm.2d.vars.iLIG127K.climo.nc'


In [ ]:
# Load Data

dat={}
sims=['pi','lig']
varns = ['PRECRC_H2Or', 'PRECRL_H2OR', 'PRECSC_H2Os', 'PRECSL_H2OS', 'PRECRC_HDOr', 'PRECRL_HDOR', 'PRECSC_HDOs', 'PRECSL_HDOS', 
         'PRECRC_H216Or', 'PRECRL_H216OR', 'PRECSC_H216Os', 'PRECSL_H216OS', 'PRECRC_H218Or', 'PRECRL_H218OR', 'PRECSC_H218Os', 'PRECSL_H218OS', 
         'PRECC', 'PRECL']

for sim in ['pi']:
    dat[sim]={}
    for varn in varns:
        dat[sim][varn]=xr.open_dataset(files[sim][varn])[varn]
        dat[sim][varn].attrs['original_time_values'] = dat[sim][varn].time
        # update time axis?
        dat[sim][varn]=dat[sim][varn].rename({'time':'month'})
        dat[sim][varn]=dat[sim][varn].assign_coords(month=[1,2,3,4,5,6,7,8,9,10,11,12])
     
for sim in ['lig']:
    dat[sim]={}
    for varn in varns:
        dat[sim][varn]=xr.open_dataset(files[sim][varn])[varn]


In [ ]:
# Calculate Climatologies

dDp={}
d18Op={}
prec={}
sims=['pi','lig']

for sim in sims:
    ptiny=1e-18;
    
    ## Precipitation
    # calculate total precip from convective and large-scale prec vars (snow+rain). convert from m/s to mm/day
    prec[sim] = (dat[sim]['PRECC'] + dat[sim]['PRECL'])*1000*60*60*24
    prec[sim].attrs['units'] = 'mm/day'
    prec[sim].attrs['long_name'] = 'total precipitation'
    prec[sim].attrs['source'] = 'PRECC + PRECL'
    # calculate precipitation weights by month
    annual_total_p = prec[sim].sum(dim="month")
    """
    if sim in ['pi']:
        annual_total_p = prec[sim].sum(dim="time")
    if sim in ['lig']:
        annual_total_p = prec[sim].sum(dim="month")
    """
    pWeights = prec[sim]/annual_total_p
    
    ## Hydrogen
    ph = dat[sim]['PRECRC_H2Or'] + dat[sim]['PRECRL_H2OR'] + dat[sim]['PRECSC_H2Os'] + dat[sim]['PRECSL_H2OS']
    pd = dat[sim]['PRECRC_HDOr'] + dat[sim]['PRECRL_HDOR'] + dat[sim]['PRECSC_HDOs'] + dat[sim]['PRECSL_HDOS']
    # replace very small ph values with a tiny value
    ph = ph.where(ph > ptiny, ptiny) 
    # turn into per mil notation
    dd = (pd/ph - 1)*1000 
    # Multiply isotope values by weights
    dDp[sim] = dd*pWeights
    
    ## Oxygen
    p16o = dat[sim]['PRECRC_H216Or'] + dat[sim]['PRECRL_H216OR'] + dat[sim]['PRECSC_H216Os'] + dat[sim]['PRECSL_H216OS']
    p18o = dat[sim]['PRECRC_H218Or'] + dat[sim]['PRECRL_H218OR'] + dat[sim]['PRECSC_H218Os'] + dat[sim]['PRECSL_H218OS']
    # replace very small ph values with a tiny value
    p16o = p16o.where(p16o > ptiny, ptiny)
    # turn into per mil notation
    do = (p18o/p16o - 1)*1000 
    # Multiply isotope values by weights
    d18Op[sim] = do*pWeights


In [ ]:
# Calculate Climatologies

dDp={}
d18Op={}
prec={}
sims=['pi','lig']

for sim in sims:
    ptiny=1e-18;
    
    ## Precipitation
    # calculate total precip from convective and large-scale prec vars (snow+rain). convert from m/s to mm/day
    prec[sim] = (dat[sim]['PRECC'] + dat[sim]['PRECL'])*1000*60*60*24
    prec[sim].attrs['units'] = 'mm/day'
    prec[sim].attrs['long_name'] = 'total precipitation'
    prec[sim].attrs['source'] = 'PRECC + PRECL'
    # calculate precipitation weights by month
    annual_total_p = prec[sim].sum(dim="month")
    """
    if sim in ['pi']:
        annual_total_p = prec[sim].sum(dim="time")
    if sim in ['lig']:
        annual_total_p = prec[sim].sum(dim="month")
    """
    pWeights = prec[sim]/annual_total_p
    
    ## Hydrogen
    ph = dat[sim]['PRECRC_H2Or'] + dat[sim]['PRECRL_H2OR'] + dat[sim]['PRECSC_H2Os'] + dat[sim]['PRECSL_H2OS']
    pd = dat[sim]['PRECRC_HDOr'] + dat[sim]['PRECRL_HDOR'] + dat[sim]['PRECSC_HDOs'] + dat[sim]['PRECSL_HDOS']
    # replace very small ph values with a tiny value
    ph = ph.where(ph > ptiny, ptiny) 
    # turn into per mil notation
    dd = (pd/ph - 1)*1000 
    # Multiply isotope values by weights
    dDp[sim] = dd*pWeights
    
    ## Oxygen
    p16o = dat[sim]['PRECRC_H216Or'] + dat[sim]['PRECRL_H216OR'] + dat[sim]['PRECSC_H216Os'] + dat[sim]['PRECSL_H216OS']
    p18o = dat[sim]['PRECRC_H218Or'] + dat[sim]['PRECRL_H218OR'] + dat[sim]['PRECSC_H218Os'] + dat[sim]['PRECSL_H218OS']
    # replace very small ph values with a tiny value
    p16o = p16o.where(p16o > ptiny, ptiny)
    # turn into per mil notation
    do = (p18o/p16o - 1)*1000 
    # Multiply isotope values by weights
    d18Op[sim] = do*pWeights


## Figs

In [ ]:
# Calculate pattern correlation
a = dDdiff.sel(lon=slice(235,275), lat=slice(10,42)).values.flatten()
b = pdiff.sel(lon=slice(235,275), lat=slice(10,42)).values.flatten()

r, p = pearsonr(a,b)

print(f"Pattern correlation: r = {r:.3f}, p = {p:.3e}")

In [ ]:
# Proxy Data
clons=[-106.5183, -111.62] 
clats=[22.5183, 27.85]
ddiff=[12.42523674, -0.67940596] #might need to recalculate based on Jess' repicks
# Model Data
lon = dat['pi']['PRECC'].lon
lat = dat['pi']['PRECC'].lat
# var specs
im=6 #start month
em=9 #end month
# plot specs
lw=1
text_kw={'color':'k', 'weight':'bold', 'size':14, 'ha':'center', 'va':'bottom'}
text_kw1={'color':'k', 'weight':'bold', 'size':12, 'ha':'left', 'va':'bottom'}
titles=np.array(['$\Delta$$\delta$D$_{precip}$', '$\Delta$Precipitation'])
# map specs
trans=ccrs.PlateCarree()
proj=ccrs.PlateCarree()
map_bnds=[-125., -85., 10., 42.]
# isotopes cmap
icmap=cmo.balance
ivmin=-3
ivmax=3
ilevels=np.linspace(ivmin, ivmax, 25)
inorm=mpl.colors.BoundaryNorm(ilevels, icmap.N)
# precip cmap
pcmap,_,_,_=get_settings(field='precip', diff=True)
pvmin=-3
pvmax=3
plevels=np.linspace(pvmin, pvmax, 25)
pnorm=mpl.colors.BoundaryNorm(plevels, pcmap.N)

# ------------------- #
#      Make Plot      #
# ------------------- #
months=['January-February-March', 'July-August-September']
t_months=months[1]

fig, ax = plt.subplots(nrows=1, ncols=2, figsize=(11,5), layout='constrained', subplot_kw={'projection': proj})
fig.text(.5,1,t_months+' LIG$-$PI Differences: UNADJUSTED', **text_kw)

#dDp
dDdiff = dDp['lig'][im:em,:,:].mean(dim="month") - dDp['pi'][im:em,:,:].mean(dim="month")
cf1=ax[0].pcolormesh(lon, lat, dDdiff, cmap=icmap, norm=inorm, transform=trans)
ax[0].scatter(x=clons, y=clats, c=ddiff,
              cmap=icmap, vmin=-5, vmax=5, alpha=1, edgecolor='k', s=150, transform=trans, zorder=100)
ax[0].text(-105.9, 23, u'12.4‰', fontsize=10, weight='bold', ha='left')
ax[0].text(-111, 28.35, u'-0.7‰', fontsize=10, weight='bold', ha='left')

# precip
pdiff = prec['lig'][im:em,:,:].mean(dim="month") - prec['pi'][im:em,:,:].mean(dim="month")
cf2=ax[1].pcolormesh(lon, lat, pdiff, cmap=pcmap, norm=pnorm, transform=trans)
ax[1].scatter(clons, clats, c='k', s=150, alpha=1, transform=trans, zorder=100)


for i in [0,1]:
    ax[i].coastlines()
    ax[i].add_feature(cfeature.BORDERS)
    ax[i].add_feature(cfeature.STATES, linewidth=0.5)
    ax[i].text(map_bnds[0], map_bnds[3]+0.5, titles[i], **text_kw1)
    ring=LinearRing(list(zip([-113., -105, -105, -113.], [18,  18,  33,  33])))
    ax[i].add_geometries([ring], crs=trans, fc='none', ec='k', lw=1, linestyle='--', zorder=9)
    ax[i].set_extent(map_bnds, crs=trans)
    if i==0:
        gl=ax[i].gridlines(crs=trans, lw=0.5, colors='black', alpha=1.0, linestyle='--', zorder=10, draw_labels=True)
        gl.top_labels=False; gl.right_labels=False
    if i==1:
        gl=ax[i].gridlines(crs=trans, lw=0.5, colors='black', alpha=1.0, linestyle='--', zorder=10, draw_labels=True)
        gl.top_labels=False; gl.left_labels=False; gl.right_labels=False

cbar_ax1 = fig.add_axes([0.05, -0.025, 0.45, 0.05])
cbar1 = fig.colorbar(cf1, ticks=[-3,-2,-1,0,1,2,3], orientation='horizontal', extend='both', cax=cbar_ax1)
cbar1.set_label(u'[‰]', weight='normal', labelpad=5, rotation=0)
cbar1.ax.tick_params(labelsize=10)
for tick in cbar1.ax.xaxis.get_major_ticks():
    tick.label1.set_fontweight('normal')

cbar_ax2 = fig.add_axes([0.535, -0.025, 0.45, 0.05])
cbar2 = fig.colorbar(cf2, ticks=[-3,-2,-1,0,1,2,3], orientation='horizontal', extend='both', cax=cbar_ax2)
cbar2.set_label('[mm day$^{-1}$]', weight='normal', labelpad=5, rotation=0)
cbar2.ax.tick_params(labelsize=10)
for tick in cbar2.ax.xaxis.get_major_ticks():
    tick.label1.set_fontweight('normal')

fig.text(0,-0.175, r'For $\mathit{unadjusted}$ iCESM1.2 LIG (127ka) output. Pattern correlation between differences for displayed domain: r $= -0.663$')

#plt.savefig("cesm1.2_LIG-PI_jas_dDp_precip.pdf")